# Supplementary analyses of the RNNs trained in PRE and POST

Networks trained with `training_PRE_POST.ipynb`; S2 uses the results of section 3 of `analyses_PRE_POST.ipynb`.
Run in the `NEURCOMP` virtual environment (`../environment.yml`).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import glob
import pickle
import platform
import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import ttest_rel, pearsonr, t as tdist

sys.path.insert(0, os.path.abspath('..'))  # func/ is in python/
from func import *

%matplotlib inline
plt.style.use('default')
plt.rc('axes', facecolor='w')
plt.rc('figure', facecolor='w')

print(platform.platform())
print(sys.executable)  # should be the NEURCOMP environment
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(device)

condtn = ['_PRE', '_POST', '_REF']  # PRE:0; POST:1; REF:2
rgb_PRE  = (0.0000, 0.4470, 0.7410)
rgb_POST = (0.8500, 0.3250, 0.0980)
rgb_REF  = (.7, .7, .7)

In [ ]:
# load the trained networks (one .tar per network) and the networks saved every savecheck epochs (.pt)
sigcat = 15
noise_val = 0
noise_str = f'{noise_val:.2f}'.replace('.', '-')
fname_train = os.path.join('..', 'rnn_simulations', 'RNN_train_PRE_POST', f'sigcat_{sigcat}' + (f'_noise_{noise_str}' if noise_val != 0 else ''))  # sigcat_X (sigcat_X_noise_X-XX if trained with noise)

file_list = sorted(glob.glob(os.path.join(fname_train, 'model_irnn_*.tar')))  # sorted by name = by network number
irnn_list = [int(os.path.basename(file).split('_')[2][:-4]) for file in file_list]  # network numbers (seeds)
file_list_epo = []
acc_train_PRE, acc_train_POST = [], []  # accuracy along training (%, last batch of each epoch)
acc_train_opt = []                      # accuracy of the optimal observer on the same batches (%)
for file in file_list:
    file_list_epo.append(sorted(glob.glob(file[:-4] + '_epo_*.pt')))  # 4-digit epochs: sorted by name = by epoch
    net = torch.load(file, weights_only=False)  # saved network: cfg and training history
    acc_train_PRE.append(net['history_train']['PRE'])
    acc_train_POST.append(net['history_train']['POST'])
    acc_train_opt.append(net['history_train']['opt'])
acc_train_PRE, acc_train_POST, acc_train_opt = np.array(acc_train_PRE), np.array(acc_train_POST), np.array(acc_train_opt)
cfg = net['cfg']  # same training configuration for all networks

N = len(file_list)
nepo = len(file_list_epo[0])  # number of saved epochs
epochs = np.array([int(f[-7:-3]) for f in file_list_epo[0]])  # saved epochs (0: untrained network)
print(f'{N} networks (sigcat = {sigcat}, noise = {noise_val}), {nepo} saved epochs each ({epochs[0]} to {epochs[-1]} epochs)')
if any(len(f) != nepo for f in file_list_epo):
    print('Warning: not the same number of saved epochs for all networks!')

In [ ]:
# configuration of the tests (all sections): each network is always tested on its own sequences
seq2test   = [4, 8, 12]   # numbers of stimuli
nseq       = 1000         # sequences per number of stimuli (each one tested in PRE and in POST)
seed_test  = 1            # sequences of network irnn: SeedSequence(seed_test, spawn_key=(irnn,)), not the training seeds
condn_test = [0, 1]       # PRE, POST
var2decode = ['cumllr', 'cumnul', 'cumsumcos', 'cumsumsin']
seeds = [int(np.random.SeedSequence(seed_test, spawn_key=(irnn,)).generate_state(1)[0]) for irnn in irnn_list]
seeds_new = [int(np.random.SeedSequence(seed_test, spawn_key=(irnn, 1)).generate_state(1)[0]) for irnn in irnn_list]  # other sequences of network irnn

## S1. Resilience to input noise

As section 2 of `analyses_PRE_POST.ipynb` (same networks, last epoch, same sequences), with noise added to the
orientation of each stimulus in the input of the network (s.d. in degrees) instead of recurrent noise. The decoded
variables and the optimal observer use the noise-free stimuli. With the same seed, the noise samples are the same for
all noise levels, only scaled by the noise level.

In [ ]:
# test the trained networks (last epoch) for each level of input noise, decode, and save
input_noise_array = np.arange(0, 41, 2)  # s.d. (degrees) of the noise on the orientation of each stimulus
fname_input = os.path.join(fname_train, 'test_input_noise.pkl')
out_input = Parallel(n_jobs=-1)(
    delayed(run_simulations)(file_list_epo[i][-1], cfg, seq2test, nseq, seeds[i], device, var2decode, condn_test,
                        input_noises=input_noise_array)
    for i in range(N))
with open(fname_input, 'wb') as f:
    pickle.dump({'out_input': out_input, 'irnn_list': irnn_list, 'file_epo': [f[-1] for f in file_list_epo],
                 'input_noise_array': input_noise_array, 'seq2test': seq2test, 'nseq': nseq, 'seed_test': seed_test,
                 'condn_test': condn_test, 'var2decode': var2decode}, f)
print(f'saved in {fname_input}')

In [ ]:
# load the saved results and gather them: (N, nnoise, 2) for PRE and POST
fname_input = os.path.join(fname_train, 'test_input_noise.pkl')
with open(fname_input, 'rb') as f:
    res_input = pickle.load(f)
out_input         = res_input['out_input']
var2decode        = res_input['var2decode']
input_noise_array = res_input['input_noise_array']

acc_input = np.array([o['pcor'] for o in out_input])  # decision accuracy
# decoding r, on all sequences, on correct sequences and on error sequences
rdec_input, rdec_input_cor, rdec_input_err = (
    {v: np.array([o[key][v] for o in out_input]) for v in var2decode} for key in ('rdec', 'rdec_cor', 'rdec_err'))
for r in (rdec_input, rdec_input_cor, rdec_input_err):
    r['compression'] = r['cumllr'] - r['cumnul']
    r['cumang'] = (r['cumsumcos'] + r['cumsumsin']) / 2  # coding of the mean orientation
print(f'{len(input_noise_array)} levels of input noise from {input_noise_array[0]} to {input_noise_array[-1]} degrees')

In [ ]:
# accuracy difference PRE - POST as a function of input noise: cluster-based permutation test on the paired difference,
# one-tailed (PRE > POST), as for the recurrent noise
acc_diff = (acc_input[:, :, 0] - acc_input[:, :, 1]) * 100  # (%)
stat_input = cluster_permutation(acc_diff, None, alpha_form=0.05, tail='right', nperm=10000, seed=1234, xvec=input_noise_array)
for c in stat_input['clusters']:
    dz = np.mean(stat_input['t'][c['idx']]) / np.sqrt(N)  # Cohen's dz of the cluster
    print(f"input noise {c['xbeg']}-{c['xend']}: PRE > POST, dz = {dz:+.2f}, p = {c['p']:.4f}")

# mean +/- s.e.m. across networks
yavg = np.mean(acc_diff, axis=0)
yerr = np.std(acc_diff, axis=0, ddof=1) / np.sqrt(N)
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.fill_between(input_noise_array, yavg - yerr, yavg + yerr, color=(.8, .8, .8), linewidth=0)
ax.plot(input_noise_array, yavg, color=(.25, .25, .25), linewidth=1.5)
ax.axhline(0, color='k', linestyle=':')
# significant clusters (p < 0.05) on the zero line
for c in stat_input['clusters']:
    if c['p'] < 0.05:
        ax.plot([c['xbeg'], c['xend']], [0, 0], color=rgb_PRE, linewidth=3)
ax.set_xlim(input_noise_array[0], input_noise_array[-1])
ax.set_xlabel('input noise (deg)')
ax.set_ylabel('accuracy difference (%)')
plt.tight_layout()

In [ ]:
# mean accuracy (PRE and POST) as a function of input noise: single networks (light), mean +/- s.e.m. (black)
acc_mean = np.mean(acc_input, axis=2)  # (N, nnoise)
yavg = np.mean(acc_mean, axis=0)
yerr = np.std(acc_mean, axis=0, ddof=1) / np.sqrt(N)
fig, ax = plt.subplots(figsize=(4.5, 3))
ax.plot(input_noise_array, acc_mean.T, color=(.8, .8, .8), linewidth=0.5)
ax.fill_between(input_noise_array, yavg - yerr, yavg + yerr, color=(.6, .6, .6), linewidth=0)
ax.plot(input_noise_array, yavg, color='k', linewidth=1.5)
ax.set_xlim(input_noise_array[0], input_noise_array[-1])
ax.set_ylim(0.5, 0.85)
ax.set_xlabel('input noise (deg)')
ax.set_ylabel('mean accuracy')
plt.tight_layout()

In [ ]:
# coding precision on the correct (solid) and error (dashed) sequences as a function of input noise
icond = 0  # PRE
rgb = (rgb_PRE, rgb_POST)[icond]
fig, axs = plt.subplots(3, 1, figsize=(4.5, 8), sharex=True)
for ax, (v, name) in zip(axs, (('cumang', 'mean orientation'), ('cumnul', 'null belief'), ('cumllr', 'true belief'))):
    y_cor = rdec_input_cor[v][:, :, icond]
    y_err = rdec_input_err[v][:, :, icond]
    ax.plot(input_noise_array, y_cor.T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
    ax.plot(input_noise_array, y_err.T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
    ax.axhline(0, color=(.8, .8, .8))
    ax.plot(input_noise_array, np.mean(y_cor, axis=0), color=rgb, linewidth=2, label='correct')
    ax.plot(input_noise_array, np.mean(y_err, axis=0), color=rgb, linewidth=2, linestyle='--', label='error')
    ax.set_ylim(-0.1, 1)
    ax.set_title(name)
    ax.set_ylabel('coding precision (r)')
    # difference between correct and error sequences, averaged over noise levels: mean +/- s.e.m. across networks
    d = np.mean(y_cor - y_err, axis=1)
    print(f'{name}: % difference between correct and error trials: {100 * np.mean(d):.3f} ± {100 * np.std(d, ddof=1) / np.sqrt(N):.3f} %')
axs[0].legend(frameon=False)
axs[-1].set_xlabel('input noise (deg)')
axs[-1].set_xlim(input_noise_array[0], input_noise_array[-1])
plt.tight_layout()

In [ ]:
# export to .mat (plots in MATLAB): (N networks, noise levels, conditions PRE and POST); indices of the clusters are 1-based
export_mat(os.path.join(fname_train, 'test_input_noise.mat'), sigcat=sigcat, irnn=np.array(irnn_list),
           input_noise_array=input_noise_array, acc=acc_input, rdec=rdec_input, rdec_cor=rdec_input_cor,
           rdec_err=rdec_input_err)#

## S2. Causal perturbation as a function of the perturbation strength

Results of section 3 of `analyses_PRE_POST.ipynb` for each noise level (perturbation strength), in PRE and in POST.
The exported `test_perturbation.mat` of section 3 already contains these values.

In [ ]:
# load the saved results and gather them
fname_pert = os.path.join(fname_train, 'test_perturbation.pkl')
with open(fname_pert, 'rb') as f:
    res_pert = pickle.load(f)
out_pert        = res_pert['out_pert']
var2decode      = res_pert['var2decode']
var2perturb     = res_pert['var2perturb']
sel_noise_array = res_pert['sel_noise_array']

# unperturbed networks: (N, 2) for PRE and POST
acc0  = np.array([o['pcor0'] for o in out_pert])
rdec0 = {v: np.array([o['rdec0'][v] for o in out_pert]) for v in var2decode}
# perturbation along the direction of each variable: (N, nlev, 2)
acc_pert  = {p: np.array([o['pcor'][p] for o in out_pert]) for p in var2perturb}
rdec_pert = {p: {v: np.array([o['rdec'][p][v] for o in out_pert]) for v in var2decode} for p in var2perturb}
for r in [rdec0] + list(rdec_pert.values()):
    r['compression'] = r['cumllr'] - r['cumnul']
    r['cumang'] = (r['cumsumcos'] + r['cumsumsin']) / 2  # coding of the mean orientation
# perturbation of the mean orientation: average of the perturbations along cumsumcos and along cumsumsin
acc_pert['cumang']  = (acc_pert['cumsumcos'] + acc_pert['cumsumsin']) / 2
rdec_pert['cumang'] = {v: (rdec_pert['cumsumcos'][v] + rdec_pert['cumsumsin'][v]) / 2 for v in rdec_pert['cumsumcos']}
# coding of the mean orientation under its own perturbation: True = coding of the perturbed component only
# (cumsumcos under the cumsumcos perturbation, cumsumsin under the cumsumsin perturbation),
# False = coding of both components (cumang), as for the other perturbations
cumang_matched = True
if cumang_matched:
    rdec_pert['cumang']['cumang'] = (rdec_pert['cumsumcos']['cumsumcos'] + rdec_pert['cumsumsin']['cumsumsin']) / 2

In [ ]:
# columns: perturbed variable, rows: accuracy (% of the unperturbed network), mean orientation coding, compression
# single networks (light), mean (thick), PRE and POST
perturb = [('cumang', 'mean orientation'), ('cumllr', 'true belief'), ('cumnul', 'null belief')]
fig, axs = plt.subplots(3, 3, figsize=(11, 9), sharex=True)
for j, (p, name) in enumerate(perturb):
    for icond, rgb in enumerate((rgb_PRE, rgb_POST)):
        rows = [acc_pert[p][:, :, icond] / acc0[:, [icond]] * 100,
                rdec_pert[p]['cumang'][:, :, icond],
                rdec_pert[p]['compression'][:, :, icond]]
        for i, y in enumerate(rows):
            axs[i, j].plot(sel_noise_array, y.T, color=(1 + np.array(rgb)) / 2, linewidth=0.5)
            axs[i, j].plot(sel_noise_array, np.mean(y, axis=0), color=rgb, linewidth=2)
    axs[0, j].set_title(name)
    axs[0, j].axhline(100, color='k', linestyle=':')
    axs[0, j].set_ylim(50, 102)
    axs[1, j].set_ylim(0, 1)
    axs[2, j].set_ylim(-0.1, 1)
    axs[2, j].set_xlabel('perturbation strength')
axs[0, 0].set_ylabel('accuracy (% unperturbed)')
axs[1, 0].set_ylabel('mean orientation\ncoding precision (r)')
axs[2, 0].set_ylabel('compression (r)')
plt.tight_layout()